# T00 — AnnData + Scanpy: Foundation

**Tools:** `anndata` · `scanpy`  
**Dataset:** PBMC 3k (Zheng et al. 2017, 10x Genomics)  
**Paper:** [Wolf et al. 2018, Genome Biology](https://doi.org/10.1186/s13059-017-1382-0)

---

## What is AnnData?

AnnData (`anndata`) is the universal data container for the scverse ecosystem — roughly the equivalent of a **SummarizedExperiment** from Bioconductor, but designed for Python. Every downstream tool (scanpy, scvi-tools, squidpy, cellrank, muon) reads and writes AnnData objects.

The key insight: a single-cell experiment is a matrix of `n_obs × n_vars` (cells × genes) plus a lot of metadata and derived quantities. AnnData stores all of this in one coherent object that can be serialized to `.h5ad` (HDF5-backed).

```
AnnData object with n_obs × n_vars = cells × genes

  .X            – main data matrix (sparse or dense), shape (n_obs, n_vars)
  .obs          – cell/observation metadata, DataFrame (n_obs × any)
  .var          – gene/feature metadata, DataFrame (n_vars × any)
  .obsm         – multidim arrays per cell  (PCA, UMAP coords, etc.)
  .varm          – multidim arrays per gene  (loadings, etc.)
  .obsp         – pairwise cell matrices     (kNN graph, distances)
  .varp         – pairwise gene matrices
  .uns           – unstructured metadata     (color palettes, PCA params, etc.)
  .layers       – additional matrices same shape as .X  (raw counts, spliced/unspliced)
  .raw           – snapshot of .X before filtering (for DE on raw counts)
```

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1       # 0=errors, 1=warnings, 2=info, 3=debug
sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'anndata {ad.__version__}  |  scanpy {sc.__version__}')

## 1. AnnData Object Anatomy

Let's build one from scratch to understand each slot before loading real data.

In [ ]:
# --- construct a minimal AnnData from scratch ---
np.random.seed(42)
n_cells, n_genes = 100, 50

# Main matrix: cells × genes (sparse is standard in practice)
import scipy.sparse as sp
X = sp.random(n_cells, n_genes, density=0.3, format='csr')

# Cell (obs) metadata
obs = pd.DataFrame({
    'cell_type': np.random.choice(['T cell', 'B cell', 'Monocyte'], n_cells),
    'n_counts': X.sum(axis=1).A1,
}, index=[f'cell_{i}' for i in range(n_cells)])

# Gene (var) metadata
var = pd.DataFrame({
    'gene_name': [f'gene_{i}' for i in range(n_genes)],
    'highly_variable': np.random.choice([True, False], n_genes),
}, index=[f'gene_{i}' for i in range(n_genes)])

adata = ad.AnnData(X=X, obs=obs, var=var)
print(adata)   # the repr shows shape, what slots are populated

In [ ]:
# Accessing slots
print('obs columns:', adata.obs.columns.tolist())
print('var columns:', adata.var.columns.tolist())
print('X type:', type(adata.X), '  shape:', adata.X.shape)

# Add a layer (e.g. raw counts before normalization)
adata.layers['raw_counts'] = adata.X.copy()

# Add obsm (multidim cell annotation, e.g. from PCA)
adata.obsm['X_pca'] = np.random.randn(n_cells, 20)

# Add uns (free-form dict)
adata.uns['experiment'] = {'protocol': 'scRNA-seq', 'organism': 'Homo sapiens'}

print('\nAfter adding slots:')
print(adata)

In [ ]:
# Slicing — just like pandas/numpy, boolean or label-based
t_cells = adata[adata.obs['cell_type'] == 'T cell']
print('T cells only:', t_cells.shape)

# Slice genes too
hvg = adata[:, adata.var['highly_variable']]
print('HVG subset:', hvg.shape)

# Slicing returns a *view* (no copy), same as pandas:
print('Is view?', t_cells.is_view)

In [ ]:
# Concatenation — common when merging samples/batches
adata2 = adata.copy()
adata2.obs['batch'] = 'batch2'
adata.obs['batch'] = 'batch1'

combined = ad.concat([adata, adata2], label='batch', keys=['b1', 'b2'])
print('Combined:', combined.shape)
print(combined.obs['batch'].value_counts())

In [ ]:
# Save and reload (the .h5ad format)
import tempfile, os
tmp = tempfile.mktemp(suffix='.h5ad')
adata.write_h5ad(tmp)
reloaded = ad.read_h5ad(tmp)
print('Reloaded:', reloaded)
os.unlink(tmp)

# For very large datasets: backed mode (disk-backed, no full load to RAM)
# adata_large = ad.read_h5ad('huge.h5ad', backed='r')  # lazy

---
## 2. The Scanpy Preprocessing Pipeline

Scanpy implements the standard scRNA-seq workflow. The function naming convention is:
- `sc.pp.*` — preprocessing (modifies the object in place)
- `sc.tl.*` — tools (adds results to `.obsm`, `.uns`, `.obs`)
- `sc.pl.*` — plotting

We'll use the canonical **PBMC 3k** dataset: 2,700 peripheral blood mononuclear cells from a healthy donor, sequenced with 10x Chromium v1.

In [ ]:
# Load the PBMC 3k dataset (downloads ~5 MB on first run)
adata = sc.datasets.pbmc3k()
print(adata)

In [ ]:
# --- QC metrics ---
# Mark mitochondrial genes (proxy for dying cells)
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# calculate_qc_metrics adds to .obs: n_genes_by_counts, total_counts,
# pct_counts_mt; and to .var: n_cells_by_counts, mean_counts, etc.
sc.pp.calculate_qc_metrics(
    adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
)

sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4, multi_panel=True,
)

## 2b. Modern QC: Doublet Detection and Ambient RNA

Two steps that are standard in any real pipeline but often missing from introductory tutorials:

### Doublet Detection
When two cells are captured in the same droplet they appear as one cell with ~2× the counts and a mixed transcriptome. Two common approaches:

| Tool | Method | When to use |
|------|--------|-------------|
| **Scrublet** | Simulates doublets by randomly mixing cell pairs; scores each cell | Fast, no GPU, good default |
| **SOLO** (`scvi-tools`) | Trains a VAE, then a classifier on simulated doublets | More accurate, especially on complex datasets |

### Ambient RNA Removal (CellBender)
Empty droplets that lysed cells leave ambient mRNA floating in solution. Every captured cell gets a small contamination from this "soup". **CellBender** (Fleming et al. 2023, Nature Methods) models and removes this using a deep generative model.

> **Important:** CellBender runs on the raw unfiltered feature-barcode matrix (from Cell Ranger's `raw_feature_bc_matrix/` output) *before* loading into Python. It outputs a cleaned `.h5` file you then load as normal. It requires a GPU for reasonable speed but has a CPU fallback.
```bash
# Typical CellBender call (run before this notebook)
cellbender remove-background \
  --input raw_feature_bc_matrix.h5 \
  --output cellbender_output.h5 \
  --expected-cells 3000 \
  --total-droplets-included 15000
```

In [ ]:
# --- Doublet Detection with Scrublet ---
# Install: pip install scrublet
import scrublet

# Load fresh PBMC 3k (needs raw counts before normalization)
adata_raw = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata_raw, min_genes=200)
sc.pp.filter_genes(adata_raw, min_cells=3)

# Scrublet simulates doublets by adding pairs of real cells,
# then scores each cell by similarity to the simulated doublets
scrub = scrublet.Scrublet(adata_raw.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

adata_raw.obs['doublet_score'] = doublet_scores
adata_raw.obs['predicted_doublet'] = predicted_doublets

print(f'Predicted doublets: {predicted_doublets.sum()} ({predicted_doublets.mean():.1%})')
scrub.plot_histogram()

In [ ]:
# --- Doublet Detection with SOLO (scvi-tools) ---
# More accurate than Scrublet, especially on complex atlas-scale data
# Trains a VAE on the data, then trains a binary classifier on simulated doublets

import scvi

scvi.model.SCVI.setup_anndata(adata_raw, layer=None)  # uses .X (raw counts)
vae = scvi.model.SCVI(adata_raw)
vae.train(max_epochs=10)  # short training; 400 epochs for production

solo = scvi.external.SOLO.from_scvi_model(vae)
solo.train(max_epochs=50)

df = solo.predict()
df['prediction'] = solo.predict(soft=False)
adata_raw.obs['solo_doublet_score'] = df['doublet']
adata_raw.obs['solo_doublet'] = df['prediction'] == 'doublet'

print(f'SOLO doublets: {adata_raw.obs["solo_doublet"].sum()} ({adata_raw.obs["solo_doublet"].mean():.1%})')

# Compare Scrublet vs SOLO agreement
import pandas as pd
agreement = pd.crosstab(
    adata_raw.obs['predicted_doublet'].rename('Scrublet'),
    adata_raw.obs['solo_doublet'].rename('SOLO'),
)
print('\nScrublet vs SOLO agreement:')
print(agreement)

In [ ]:
# --- Apply doublet filter before downstream analysis ---
# Use SOLO predictions (or Scrublet if scvi-tools not installed)
doublet_col = 'solo_doublet' if 'solo_doublet' in adata_raw.obs else 'predicted_doublet'
n_before = adata_raw.n_obs
adata_clean = adata_raw[~adata_raw.obs[doublet_col]].copy()
print(f'Removed {n_before - adata_clean.n_obs} doublets, {adata_clean.n_obs} cells remain')

# Visualize doublet score distribution on UMAP
# (run the full scanpy pipeline first, then color by doublet_score)
# sc.pl.umap(adata, color='doublet_score', color_map='Reds')

In [ ]:
# --- Filter cells and genes ---
# Remove low-quality cells
adata = adata[adata.obs['n_genes_by_counts'] < 2500, :]   # doublets
adata = adata[adata.obs['pct_counts_mt'] < 5, :]          # dead cells
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print('After filtering:', adata.shape)

In [ ]:
# --- Normalization ---
# 1. Library-size normalize to 10,000 counts per cell ("CPM-style")
# 2. Log1p: log(x+1) — compresses dynamic range, makes variance more stable
# Save raw counts in a layer first
adata.layers['counts'] = adata.X.copy()          # preserve for scVI later
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# --- Highly Variable Genes (HVGs) ---
# Identify genes with high variance relative to mean (biological signal > noise)
# These are used for PCA, neighbors, UMAP — not for DE (we use .raw for that)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
print(f'{adata.var.highly_variable.sum()} HVGs out of {adata.n_vars} genes')

In [ ]:
# Snapshot the normalized+log1p matrix before HVG subsetting
# adata.raw stores .X and .var at this point for later DE retrieval
adata.raw = adata

# Subset to HVGs
adata = adata[:, adata.var.highly_variable]

# Scale each gene to zero mean, unit variance (clips at max_value)
# This ensures PCA isn't dominated by high-count genes
sc.pp.scale(adata, max_value=10)

## 3. Dimensionality Reduction, Neighbors, Clustering

In [ ]:
# PCA: linear dimensionality reduction
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)
# Results stored in adata.obsm['X_pca'] and adata.varm['PCs']

In [ ]:
# k-nearest neighbor graph in PCA space
# This graph is the backbone for both UMAP and Leiden clustering
# Results: adata.obsp['connectivities'] and adata.obsp['distances']
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Non-linear 2D embedding for visualization
sc.tl.umap(adata)

# Leiden community detection (graph-based clustering)
# resolution: higher = more/smaller clusters
sc.tl.leiden(adata, resolution=0.5, random_state=0)

sc.pl.umap(adata, color=['leiden', 'n_genes_by_counts'], legend_loc='on data')

## 4. Cell Type Annotation

Scanpy has several ways to annotate clusters. The standard manual approach:
1. Find marker genes per cluster (`sc.tl.rank_genes_groups`)
2. Inspect known markers with dotplots/violin plots
3. Assign cell type labels to `.obs`

In [ ]:
# Rank genes for each cluster vs. all others (Wilcoxon rank-sum)
# Uses .raw (log-normalized, full gene set) by default
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# Known PBMC marker genes
marker_genes = {
    'CD14+ Monocytes': ['CD14', 'LYZ'],
    'NK cells':        ['GNLY', 'NKG7'],
    'CD8+ T cells':    ['CD8A'],
    'CD4+ T cells':    ['IL7R', 'CCR7'],
    'B cells':         ['MS4A1'],
    'Dendritic cells': ['FCER1A', 'CST3'],
    'Megakaryocytes':  ['PPBP'],
}

sc.pl.dotplot(adata, marker_genes, groupby='leiden', standard_scale='var')

In [ ]:
# Assign cell type labels (manual annotation based on dot plot)
cell_type_map = {
    '0': 'CD4+ T',  '1': 'CD14+ Mono', '2': 'CD4+ T',
    '3': 'NK',      '4': 'CD8+ T',     '5': 'B cell',
    '6': 'CD14+ Mono', '7': 'DC',      '8': 'Mega',
}
adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map).astype('category')
sc.pl.umap(adata, color='cell_type', legend_loc='on data', title='PBMC 3k — Cell types')

## 5. Key Scanpy Plots Reference

Scanpy has a rich plotting API. The most-used ones:

In [ ]:
# Violin: per-cluster gene expression distribution
sc.pl.violin(adata, ['IL7R', 'NKG7', 'MS4A1'], groupby='cell_type', rotation=45)

# Stacked violin (cleaner for many genes × many groups)
sc.pl.stacked_violin(
    adata,
    ['IL7R', 'NKG7', 'MS4A1', 'CD14', 'CD8A'],
    groupby='cell_type', swap_axes=True,
)

# Heatmap of mean expression per group
sc.pl.matrixplot(
    adata,
    ['IL7R', 'NKG7', 'MS4A1', 'CD14', 'CD8A'],
    groupby='cell_type', standard_scale='var',
)

In [ ]:
# Save the processed object for downstream notebooks
adata.write_h5ad('../data/pbmc3k_processed.h5ad')
print('Saved to ../data/pbmc3k_processed.h5ad')
print(adata)

---
## Summary

| Step | Function | Result stored in |
|------|----------|------------------|
| QC metrics | `sc.pp.calculate_qc_metrics` | `.obs`, `.var` |
| Normalize | `sc.pp.normalize_total` + `sc.pp.log1p` | `.X` |
| HVGs | `sc.pp.highly_variable_genes` | `.var.highly_variable` |
| Raw snapshot | `adata.raw = adata` | `.raw` |
| Scale | `sc.pp.scale` | `.X` |
| PCA | `sc.tl.pca` | `.obsm['X_pca']`, `.varm['PCs']` |
| kNN graph | `sc.pp.neighbors` | `.obsp['connectivities']`, `.obsp['distances']` |
| UMAP | `sc.tl.umap` | `.obsm['X_umap']` |
| Clustering | `sc.tl.leiden` | `.obs['leiden']` |
| DE markers | `sc.tl.rank_genes_groups` | `.uns['rank_genes_groups']` |

**Next:** T01 — scvi-tools for deep generative modeling and batch integration.